# Testing for differences in power scaling according to different contexts

In [1]:
import os
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from scipy.stats import f

import plotly.express as px
import plotly.graph_objects as go

from scipy.stats import f_oneway as ftest

import warnings; warnings.filterwarnings('ignore')

In [2]:
DATA_PATH = 'reports'

PARAMETERS_PATH = os.path.join(DATA_PATH, 'all.csv')
# PARAMETERS_PATH = './ignore/av-params-20250825.csv'

vis_path = DATA_PATH.replace('.csv', '.html')

In [3]:
subcorpora_col = 'context_type'

In [4]:
# to rename the corpus correctly . . . 
def lang(x):
    result = str(x)
    
    if '-DISPEL-' in result:
        result = 'DISPEL'
    
    if result.startswith('GCSAusE'):
        result = 'GCSAusE'
    
    if result.startswith('se'):
        result = 'CORAAL'
        
    if result.startswith('call'):
        result = x.split('-')[0]
    
    if result.startswith('MICASE'):
        result = 'MICASE'
    
    if result.startswith('instruction'):
        result = x.split('-')[1]
    
    if result.startswith('CABNC'):
        result = 'CABNC'
        
    if result.startswith('SBCSAE'):
        result = 'SBCSAE'
        
    if result.startswith('SCoSE'):
        result = 'SCoSE'
        
    if result.startswith('candor'):
        result = 'CANDOR'
    
    if result.startswith('grace'):
        result = 'Miao-fNIRS'
    
    return result

In [25]:
def context(x):
    context = None
    if 'callfriend' in x:
        context = 'friend'

    if 'callhome' in x:
        context = 'family'

    if ('frederiksen' in x.lower()) or ('graesser' in x.lower()):
        context = 'tutoring'

    if 'disepel' in x.lower():
        context = 'gaming tutorial'

    if 'med_school' in x.lower():
        context = 'group learning'

    if 'MICASE' in x:
        # context = x.split('-')[1][:3]
        context = x.split('-')[-1]

    if ('candor' in x.lower()) or ('grace' in x.lower()):
        context = 'first meeting'

    return context

#### Loading Results

In [6]:
results = pd.read_csv(PARAMETERS_PATH)

In [7]:
results.isna().sum()

corpus     0
cid        0
speaker    0
self       0
a          0
b          0
k          0
dtype: int64

In [8]:
results.isna().mean()

corpus     0.0
cid        0.0
speaker    0.0
self       0.0
a          0.0
b          0.0
k          0.0
dtype: float64

In [9]:
(results['b'].abs() == np.inf).sum(), (results['b'].abs() == np.inf).mean()

(np.int64(0), np.float64(0.0))

In [10]:
results.head()

,corpus,cid,speaker,self,a,b,k
0,CABNC,CABNC-KB0RE000-KB0,CABNC-CABNC-KB0RE000-KB0-KB0PSUN,False,2.240313e-04,-0.727759,1066
1,CABNC,CABNC-KB0RE000-KB0,CABNC-CABNC-KB0RE000-KB0-KB0PSUN,True,1.439414e-08,0.363082,136
2,CABNC,CABNC-KB0RE000-KB0,CABNC-CABNC-KB0RE000-KB0-PS002,False,1.320233e-03,-0.615558,3107
3,CABNC,CABNC-KB0RE000-KB0,CABNC-CABNC-KB0RE000-KB0-PS002,True,1.493614e-03,-0.797152,2556
4,CABNC,CABNC-KB0RE000-KB0,CABNC-CABNC-KB0RE000-KB0-PS006,False,5.985318e-04,-0.503047,2966


In [13]:
sel = results['b'].isna() | (results['b'].abs() == np.inf) 
sel.sum(), sel.mean(), (~sel).sum()

(np.int64(0), np.float64(0.0), np.int64(23894))

In [14]:
results['self'].value_counts()

self
False    12604
True     11290
Name: count, dtype: int64

In [15]:
results['cid'].loc[sel].nunique()

0

In [17]:
# is_null = np.array(['null-' in x for x in tqdm(results['cid'])])
# is_null = results['null']
# is_null.sum()

Adding context names . . .

In [26]:
results[subcorpora_col] = [context(x) for x in tqdm(results['cid'])]

  0%|          | 0/23894 [00:00<?, ?it/s]

In [27]:
results[subcorpora_col].value_counts()

context_type
first meeting            6624
family                    727
les                       624
lel                       407
student_presentations     341
colloquia                 245
office_hours              241
discussion                237
seminars                  190
friend                    183
meetings                  172
service_encounters        168
labs                      132
study_groups               90
defense                    63
tours                      54
group learning             34
advising                   26
tutoring                   20
interviews                 12
Name: count, dtype: int64

In [28]:
results.head()

,corpus,cid,speaker,self,a,b,k,context_type
0,CABNC,CABNC-KB0RE000-KB0,CABNC-CABNC-KB0RE000-KB0-KB0PSUN,False,2.240313e-04,-0.727759,1066,None
1,CABNC,CABNC-KB0RE000-KB0,CABNC-CABNC-KB0RE000-KB0-KB0PSUN,True,1.439414e-08,0.363082,136,None
2,CABNC,CABNC-KB0RE000-KB0,CABNC-CABNC-KB0RE000-KB0-PS002,False,1.320233e-03,-0.615558,3107,None
3,CABNC,CABNC-KB0RE000-KB0,CABNC-CABNC-KB0RE000-KB0-PS002,True,1.493614e-03,-0.797152,2556,None
4,CABNC,CABNC-KB0RE000-KB0,CABNC-CABNC-KB0RE000-KB0-PS006,False,5.985318e-04,-0.503047,2966,None


### Removing null values and unknown context types

In [29]:
results = results.loc[
    (~results[subcorpora_col].isna())
    # & (~is_null)
    & (~sel)
]
results.shape

(10590, 8)

## Differences between context types

### Self

In [30]:
sel_ = results['self']
sel_.sum()

np.int64(5038)

In [31]:
# means
df = results[[subcorpora_col, 'b']].loc[sel_].groupby(by=[subcorpora_col]).agg('mean').reset_index()

# var
df['s2'] = results[[subcorpora_col, 'b']].loc[sel_].groupby(by=[subcorpora_col]).agg('var').reset_index()['b']

# wj for welch
df['wj'] = results[[subcorpora_col, 'b']].loc[sel_].groupby(by=[subcorpora_col]).agg(lambda x: len(x)/x.var()).reset_index()['b']

# n
df['n'] = results[[subcorpora_col, 'b']].loc[sel_].groupby(by=[subcorpora_col]).agg('count').reset_index()['b']

# SSW
df['ssw'] = results[[subcorpora_col, 'b']].loc[sel_].groupby(by=[subcorpora_col]).agg(lambda x: ((x-x.mean())**2).sum()).reset_index()['b']

# SSB
df['ssb'] = (df['b'] - df['b'].mean())**2

In [32]:
df.head(n=100)

,context_type,b,s2,wj,n,ssw,ssb
0,advising,-0.278311,0.033039,393.475020,13,0.396467,0.008647
1,colloquia,-0.345494,1.501936,66.580736,100,148.691657,0.000666
2,defense,0.077508,1.617506,16.074130,26,40.437647,0.201429
3,discussion,-0.189503,2.544577,38.906265,99,249.368580,0.033050
4,family,-0.432396,0.055077,6499.955189,358,19.662597,0.003733
5,first meeting,-0.497427,0.026498,124988.394360,3312,87.736402,0.015908
6,friend,-0.431379,0.132452,679.493295,90,11.788196,0.003609
7,group learning,-0.589381,0.763379,18.339520,14,9.923924,0.047559
8,interviews,-0.347725,0.003268,1836.146058,6,0.016339,0.000556
9,labs,-0.534500,2.396790,24.616263,59,139.013791,0.026634


And the actual F-test calculations

In [33]:
k = len(df)
w = df['wj'].sum()
mu_ = ((df['wj'] * df['b'])/w).sum()

In [34]:
df['inv_nj'] = 1/(df['n']-1)
df['p_wj'] = (1 - (df['wj']/w))**2

In [35]:
num = (1/(k-1)) * df['wj'] * ((df['b']-mu_)**2)
num = num.sum()

In [36]:
denom = 1 + ((2*(k-2))/((k**2)-1)) * (df['inv_nj'] * df['p_wj']).sum()

In [37]:
dof = ((k**2)-1) / (3 * (df['inv_nj'] * df['p_wj']).sum())

In [38]:
F = num/denom
F

np.float64(7.229073695527502)

In [39]:
F, k-1, dof, 1-f.cdf(F,k-1,dof)

(np.float64(7.229073695527502),
 19,
 np.float64(190.15658901567144),
 np.float64(2.0650148258027912e-14))

#### Save table

In [40]:
for col in [c for c in list(df) if (c != 'context_type')]:
    df[col] = [
        np.format_float_scientific(x,precision=2) if x.__abs__() < .001
        else str(np.around(x, decimals=3))
        for x in df[col]
    ]

In [41]:
report_name = 'context-self-alpha-table.txt'

In [42]:
with open(report_name, 'w') as fi:
    txt =  df[[col for col in list(df) if col not in ['ssw', 'ssb', 'wj', 'inv_nj', 'p_wj']]].to_latex(index=False).replace('\\toprule', '\\hline').replace('\\midrule', '\\hline\\hline').replace('\\bottomrule', '\\hline')

    txt = txt.replace('\\\\', '\\\\\hline').replace('|', '$|$').replace('_delta', ' $\Delta$')
    fi.write(txt)
    fi.close()

### Other

Are there individual differences in the degree to which individual speakers influence their conversation partners? We replicate the same procedure in the "self" approach above on data for exponential decay rate ($\alpha$) for the influence of an utterance on the "other".

In [43]:
sel_ = ~results['self']
sel_.sum()

np.int64(5552)

In [44]:
# means
df = results[[subcorpora_col, 'b']].loc[sel_].groupby(by=[subcorpora_col]).agg('mean').reset_index()

# var
df['s2'] = results[[subcorpora_col, 'b']].loc[sel_].groupby(by=[subcorpora_col]).agg('var').reset_index()['b']

# wj for welch
df['wj'] = results[[subcorpora_col, 'b']].loc[sel_].groupby(by=[subcorpora_col]).agg(lambda x: len(x)/x.var()).reset_index()['b']

# n
df['n'] = results[[subcorpora_col, 'b']].loc[sel_].groupby(by=[subcorpora_col]).agg('count').reset_index()['b']

# SSW
df['ssw'] = results[[subcorpora_col, 'b']].loc[sel_].groupby(by=[subcorpora_col]).agg(lambda x: ((x-x.mean())**2).sum()).reset_index()['b']

# SSB
df['ssb'] = (df['b'] - (df['b'].mean()))**2

In [45]:
df.head(n=100)

,context_type,b,s2,wj,n,ssw,ssb
0,advising,-0.093968,0.117323,110.804885,13,1.407880,0.001603
1,colloquia,0.010924,0.219113,661.760358,145,31.552207,0.021004
2,defense,0.051448,0.186538,198.350818,37,6.715374,0.034393
3,discussion,-0.099826,0.085859,1607.279216,138,11.762735,0.001168
4,family,-0.395400,0.041222,8951.433497,369,15.169861,0.068328
5,first meeting,-0.481906,0.022145,149561.828473,3312,73.321061,0.121036
6,friend,-0.357906,0.063249,1470.386087,93,5.818880,0.050132
7,group learning,0.069200,1.544640,12.947997,20,29.348169,0.041292
8,interviews,-0.300603,0.001987,3020.244641,6,0.009933,0.027755
9,labs,-0.051269,0.109706,665.415257,73,7.898827,0.006845


And the actual F-test calculations

In [46]:
k = len(df)
w = df['wj'].sum()
mu_ = ((df['wj'] * df['b'])/w).sum()

In [47]:
df['inv_nj'] = 1/(df['n']-1)
df['p_wj'] = (1 - (df['wj']/w))**2

In [48]:
num = (1/(k-1)) * df['wj'] * ((df['b']-mu_)**2)
num = num.sum()

In [49]:
denom = 1 + ((2*(k-2))/((k**2)-1)) * (df['inv_nj'] * df['p_wj']).sum()

In [50]:
dof = ((k**2)-1) / (3 * (df['inv_nj'] * df['p_wj']).sum())

In [51]:
F = num/denom
F

np.float64(157.08009696541748)

In [52]:
F, k-1, dof, 1-f.cdf(F,k-1,dof)

(np.float64(157.08009696541748),
 19,
 np.float64(218.34422854201588),
 np.float64(1.1102230246251565e-16))

#### Save table

In [53]:
for col in [c for c in list(df) if (c != 'context_type')]:
    df[col] = [
        np.format_float_scientific(x,precision=2) if x.__abs__() < .001
        else str(np.around(x, decimals=3))
        for x in df[col]
    ]

In [54]:
report_name = 'context-self-other-table.txt'

In [55]:
with open(report_name, 'w') as fi:
    txt =  df[[col for col in list(df) if col not in ['ssw', 'ssb', 'wj', 'inv_nj', 'p_wj']]].to_latex(index=False).replace('\\toprule', '\\hline').replace('\\midrule', '\\hline\\hline').replace('\\bottomrule', '\\hline')

    txt = txt.replace('\\\\', '\\\\\hline').replace('|', '$|$').replace('_delta', ' $\Delta$')
    fi.write(txt)
    fi.close()